# Workspace RayCluster + job client — Ray Train

Same pattern as Topics 1–2: `Cluster` + `job_client`, with the workshop **2×GPU** cluster shape.

Runs `train_fashion_mnist.py` — Ray Train `TorchTrainer` on FashionMNIST across two GPU workers.

`ScalingConfig(num_workers=2, use_gpu=True)` must match the RayCluster worker/GPU count.

Workers need network egress (or pre-cached data) to download FashionMNIST on first run.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from workshop_bootstrap import setup_workshop_path

REPO_ROOT = setup_workshop_path()
print(f"Repo root: {REPO_ROOT}")

## Authenticate

Same credentials as Topic 1 — OpenShift Console → **Copy login command** → Display token.


In [ ]:
from kube_authkit import AuthConfig, get_k8s_client
from codeflare_sdk import set_api_client, list_local_queues, view_clusters

OPENSHIFT_SERVER = "https://api.YOUR_CLUSTER:6443"
OPENSHIFT_TOKEN = "REPLACE_WITH_YOUR_TOKEN"

auth_config = AuthConfig(
    method="openshift",
    k8s_api_host=OPENSHIFT_SERVER.strip(),
    token=OPENSHIFT_TOKEN.strip(),
    verify_ssl=False,
)
set_api_client(get_k8s_client(config=auth_config))

NAMESPACE = "ray-workshop"
LOCAL_QUEUE = "ray-workshop-queue"

list_local_queues(NAMESPACE)

## Create the RayCluster

Two workers with 1 GPU each (workshop default via `workshop_cluster_configuration`).

In [ ]:
from codeflare_sdk import Cluster
from workshop_bootstrap import workshop_cluster_configuration

cluster = Cluster(
    workshop_cluster_configuration(
        name="ray-workshop-train",
        namespace=NAMESPACE,
        local_queue=LOCAL_QUEUE,
    )
)

cluster.apply()
cluster.wait_ready()
cluster.details()

## Submit Ray Train job

Installs `torch` / `torchvision` via `runtime_env.pip` if the Ray image does not already include them. Prefer a CUDA-capable Ray image from [Supported Configurations](https://access.redhat.com/articles/6856871) when possible.

In [ ]:
import time

client = cluster.job_client

submission_id = client.submit_job(
    entrypoint="python extras/scripts/train_fashion_mnist.py",
    runtime_env={
        "working_dir": str(REPO_ROOT),
        "pip": ["torch", "torchvision"],
        "env_vars": {
            "NUM_EPOCHS": "3",
            "NUM_WORKERS": "2",
        },
    },
)
print(f"Submitted: {submission_id}")

terminal = {"SUCCEEDED", "FAILED", "STOPPED"}
deadline = time.time() + 1800

while time.time() < deadline:
    status = client.get_job_status(submission_id)
    print(f"Job {submission_id}: {status}")
    if str(status).split(".")[-1].upper() in terminal:
        break
    time.sleep(20)
else:
    raise TimeoutError(f"Timed out waiting for job {submission_id}")

print(client.get_job_logs(submission_id))

## Observe

Open the Ray Dashboard from `view_clusters()` → **Jobs** for training progress.

In [ ]:
view_clusters(NAMESPACE)

## Tear down

In [ ]:
cluster.down()
print("Cluster down.")

## Verify

1. Job reaches `SUCCEEDED`.
2. Logs show epoch loss lines and `Done. Ray Train FashionMNIST finished successfully.`
3. If the job stays PENDING, check that `NUM_WORKERS` / `ScalingConfig.num_workers` matches GPU workers on the cluster (workshop: 2).